# 05_completions

Stage 5 of the `codes/mcp/` track: **argument completions** on the same
**FastMCP + Pydantic** stack as stages 1–4.

Notebook path: `codes/mcp/05_completions/qa.ipynb`. File I/O for this
stage stays in this directory (`server.py`, `client.py`). Shared deps live in
`codes/requirements.txt` (one file for the whole `codes/` lab tree).

Install once (venv recommended), from this stage directory:

```bash
pip install -r ../../requirements.txt
```

Run the next cell first to bind `ROOT`. Edit only under `# answer`. Pre-filled
setup code stays as-is. Get each cell running before moving on.

Architecture to keep in your head:

`@mcp.completion` handler → server advertises `completions` capability →
host sends `completion/complete` with a `PromptReference` or
`ResourceTemplateReference` + `{name, value}` → handler returns candidate
strings (or a `Completion` with `total` / `has_more`). Return `None` for
unhandled refs (empty list, not an error). Multi-arg prompts pass already
filled values as `context_arguments`.

This stays protocol-honest on `2026-07-28` — no sampling back-channel, no
deprecated protocol logging. Completions are host-driven autocomplete for
prompt args and URI-template placeholders.

FastMCP's Client is async — cells below use top-level `await` (Jupyter /
IPython). From a plain script, wrap the same coroutine in `asyncio.run(...)`.


In [1]:
from pathlib import Path

def lab_root() -> Path:
    """Directory that contains this qa.ipynb."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "05_completions" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "mcp" / "05_completions"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    raise FileNotFoundError("cannot locate codes/mcp/05_completions")

ROOT = lab_root()
print(ROOT)


/workspace/AEFS-Notes/codes/mcp/05_completions


## 1. Read a host-facing completion/complete shape

`payload` is what a host sends / receives for argument completion. Print the
request `ref.type` and `argument.name`, then the response `values` list.
Confirm the prefix `"n"` yields only themes that start with `n`.


In [2]:
payload = {
    "request": {
        "ref": {"type": "ref/prompt", "name": "essay"},
        "argument": {"name": "genre", "value": "n"},
    },
    "response": {
        "completion": {
            "values": ["nature"],
            "total": None,
            "hasMore": None,
        }
    },
}

# answer

ref_type = payload["request"]["ref"]["type"]
arg_name = payload["request"]["argument"]["name"]
values = payload["response"]["completion"]["values"]
print(ref_type)
print(arg_name)
print(values)
assert ref_type == "ref/prompt"
assert arg_name == "genre"
assert values == ["nature"]
assert all(v.startswith("n") for v in values)


ref/prompt
genre
['nature']


## 2. Tiny FastMCP server with a completion handler

Create a local `FastMCP("qa-complete")` with:

1. a prompt `greet(name: str) -> str` that returns `f"hello {name}"`
2. `@server.completion` that, for `PromptReference(name="greet")` and
   `argument.name == "name"`, returns names from
   `["ada", "alan", "grace", "linus"]` filtered by `argument.value` prefix

Drive it with `Client(server)`. Print:

1. negotiated `protocol_version` (expect `2026-07-28`)
2. whether `server_capabilities.completions` is present (`True`)
3. `complete(PromptReference(name="greet"), {"name": "name", "value": "a"}).values`


In [3]:
from typing import Annotated

from fastmcp import Client, FastMCP
from mcp_types import PromptReference
from pydantic import Field

# answer

server = FastMCP("qa-complete")
NAMES = ["ada", "alan", "grace", "linus"]

@server.prompt
def greet(name: Annotated[str, Field(description="Who to greet")]) -> str:
    """Tiny greeting prompt."""
    return f"hello {name}"

@server.completion
def complete(ref, argument, context):
    if isinstance(ref, PromptReference) and ref.name == "greet":
        if argument.name == "name":
            prefix = argument.value or ""
            return [n for n in NAMES if n.startswith(prefix)]
    return None

async with Client(server) as client:
    print("protocol:", client.protocol_version)
    caps = client.server_capabilities
    print("completions:", caps.completions is not None if caps else False)
    result = await client.complete(
        PromptReference(name="greet"),
        argument={"name": "name", "value": "a"},
    )
    print("values:", result.values)


protocol: 2026-07-28
completions: True
values: ['ada', 'alan']


## 3. Drive the lab server: prompt + resource completions

Import `mcp` from `server.py` in this directory. With `Client(mcp)`:

1. print `protocol_version` and `server_info.name`
2. print whether `completions` capability is advertised
3. complete `essay` / `genre` with value `"s"` — print values
4. complete `essay` / `topic` with value `"m"` and
   `context_arguments={"genre": "sci-fi"}` — print values
5. complete `note://{slug}` / `slug` with value `"a"` — print values,
   `total`, and `has_more`

Same Client entry as earlier stages — only the request surface grows to
`completion/complete`.


In [4]:
import sys

sys.path.insert(0, str(ROOT))
from fastmcp import Client
from mcp_types import PromptReference, ResourceTemplateReference
from server import mcp

# answer

async with Client(mcp) as client:
    print("protocol:", client.protocol_version)
    print("server:", client.server_info.name if client.server_info else None)
    caps = client.server_capabilities
    print("completions:", caps.completions is not None if caps else False)

    genre = await client.complete(
        PromptReference(name="essay"),
        argument={"name": "genre", "value": "s"},
    )
    print("genre:", genre.values)

    topic = await client.complete(
        PromptReference(name="essay"),
        argument={"name": "topic", "value": "m"},
        context_arguments={"genre": "sci-fi"},
    )
    print("topic:", topic.values)

    slug = await client.complete(
        ResourceTemplateReference(uri="note://{slug}"),
        argument={"name": "slug", "value": "a"},
    )
    print("slug:", slug.values, "total=", slug.total, "has_more=", slug.has_more)


protocol: 2026-07-28
server: lab-completions
completions: True
genre: ['sci-fi']
topic: ['mars']
slug: ['alpha', 'atlas', 'aurora'] total= 3 has_more= False


## 4. Unhandled references return an empty completion

Build a local `FastMCP("qa-empty")` with `@ext.completion` that always
returns `None`. Complete a `PromptReference(name="missing")` with any
argument and print `.values` — expect `[]`, not an exception.

Extension point: one handler covers every prompt / template the server owns;
returning `None` keeps unknown refs quiet for the host.


In [5]:
# answer

ext = FastMCP("qa-empty")

@ext.completion
def complete(ref, argument, context):
    return None

async with Client(ext) as client:
    result = await client.complete(
        PromptReference(name="missing"),
        argument={"name": "x", "value": ""},
    )
    print(result.values)


[]
